# Lab 6: Hybrid Search with Neo4jContextProvider

In this lab, you will combine vector and fulltext search into a single hybrid provider that captures both semantic similarity and keyword precision.

## What you will learn

- How to configure `Neo4jContextProvider` with `index_type="hybrid"`
- That hybrid search requires both a vector index and a fulltext index
- How hybrid search combines scores from both search strategies
- When hybrid search is the right choice

## Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

provider_name = os.getenv("LLM_PROVIDER", "openai").lower()
if provider_name == "azure":
    assert os.getenv("AZURE_OPENAI_API_KEY"), "AZURE_OPENAI_API_KEY not set in .env file"
    assert os.getenv("AZURE_OPENAI_ENDPOINT"), "AZURE_OPENAI_ENDPOINT not set in .env file"
else:
    assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set in .env file"
assert os.getenv("NEO4J_URI"), "NEO4J_URI not set in .env file"
print("Environment loaded successfully")

## Import Dependencies

In [ ]:
from llm_provider import get_client, get_embedder
from agent_framework_neo4j import Neo4jContextProvider, Neo4jSettings

## Load Settings and Create Embedder

Hybrid search requires an embedder (for the vector component).

In [ ]:
neo4j_settings = Neo4jSettings()
embedder = get_embedder()

## Create the Hybrid Provider

Configure `Neo4jContextProvider` with:
- `index_name` set to `neo4j_settings.vector_index_name` (for vector search)
- `index_type="hybrid"`
- `fulltext_index_name` set to `neo4j_settings.fulltext_index_name` (for fulltext search)
- The embedder
- `top_k=5`
- A `context_prompt`

The key difference from vector-only search is the `fulltext_index_name` parameter.

In [ ]:
provider = Neo4jContextProvider(
    uri=neo4j_settings.uri,
    username=neo4j_settings.username,
    password=neo4j_settings.get_password(),
    index_name=neo4j_settings.vector_index_name,
    index_type="hybrid",
    fulltext_index_name=neo4j_settings.fulltext_index_name,
    embedder=embedder,
    top_k=5,
    context_prompt=(
        "## Movie Knowledge Graph Context\n"
        "Use the following movie information from the knowledge graph "
        "to answer questions:"
    ),
)

## Run the Agent

This query benefits from both search strategies:
- **Vector search** finds movies semantically related to "sci-fi movies about dreams"
- **Fulltext search** finds plots containing "Christopher Nolan" as a literal term

In [ ]:
async with provider:
    client = get_client()

    agent = client.as_agent(
        name="movie-hybrid-agent",
        instructions=(
            "You are a movie recommendation assistant. Answer questions "
            "using the movie data provided in your context."
        ),
        context_providers=[provider],
    )

    session = agent.create_session()

    query = "I'm looking for Christopher Nolan sci-fi movies about dreams"
    print(f"User: {query}\n")
    print("Answer: ", end="", flush=True)
    response = await agent.run(query, session=session)
    print(response.text)
    print()

## Summary

Hybrid search combines vector and fulltext search by running both and merging the results. It requires a vector index, a fulltext index, and an embedder. This mode catches both semantic and keyword matches, making it the most versatile option when users mix conceptual and specific queries.

**What's next:** In the next module, you will give agents persistent memory using the `neo4j-agent-memory` package.